# 🦜 Agente Pandas con LangChain + Mistral

## ¿Qué es `invoke`?
```
agent.invoke({"input": "tu pregunta"})
    │
    ├─ LangChain construye el prompt automáticamente
    ├─ El agente decide qué herramienta usar (pandas en este caso)
    ├─ Ejecuta código Python sobre el DataFrame
    └─ Devuelve la respuesta en lenguaje natural
```

## ¿Qué es `create_pandas_dataframe_agent`?
Crea un agente que puede ejecutar código pandas sobre tu DataFrame para responder preguntas.
No necesitas escribir ningún query — el agente lo genera solo.

---

## CELDA 1 — Instalación
> Reinicia el runtime cuando termine.

In [1]:
!pip install -q langchain langchain-mistralai langchain-experimental tabulate
print('✅ Listo. Reinicia: Entorno de ejecución → Reiniciar sesión')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Listo. Reinicia: Entorno de ejecución → Reiniciar sesión


## CELDA 2 — Imports

In [2]:
import pandas as pd
import numpy as np
from google.colab import files, userdata

from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

print('✅ Imports correctos')
print(f'   pandas {pd.__version__}')

✅ Imports correctos
   pandas 2.2.2


## CELDA 3 — Cargar el dataset

In [4]:
# Sube tu CSV cuando aparezca el botón
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename, encoding='latin-1')

print(f'✅ Dataset cargado: {filename}')
print(f'   {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

Saving archive.zip to archive (1).zip
✅ Dataset cargado: archive (1).zip
   2,823 filas × 25 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## CELDA 4 — Limpieza rápida con pandas

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Rellenar NaN
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])
    df[col] = df[col].str.strip().str.upper()

df = df.drop_duplicates()

print(f'✅ Limpieza completa')
print(f'   Filas: {len(df):,}')
print(f'   NaN restantes: {df.isnull().sum().sum()}')
print(f'   Columnas: {df.columns.tolist()}')

✅ Limpieza completa
   Filas: 2,823
   NaN restantes: 0
   Columnas: ['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'addressline2', 'city', 'state', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize']


## CELDA 5 — Crear el LLM con Mistral
> Agrega tu API key en **Secrets** (🔑) con el nombre `MISTRAL_KEY`

In [6]:
MISTRAL_API_KEY = userdata.get('MISTRAL_KEY')

# Inicializar el modelo de Mistral para LangChain
llm = ChatMistralAI(
    model='mistral-large-latest',
    api_key=MISTRAL_API_KEY,
    temperature=0       # 0 = respuestas deterministas, ideal para análisis de datos
)

print('✅ Modelo Mistral listo')
print(f'   Modelo: mistral-large-latest')
print(f'   Temperature: 0')

✅ Modelo Mistral listo
   Modelo: mistral-large-latest
   Temperature: 0


## CELDA 6 — Crear el agente pandas

```
create_pandas_dataframe_agent
    │
    ├─ llm            → el modelo que razona (Mistral)
    ├─ df             → tu DataFrame
    ├─ agent_type     → OPENAI_FUNCTIONS usa function calling
    ├─ verbose        → True muestra el razonamiento paso a paso
    └─ allow_dangerous_code → necesario para ejecutar pandas
```

In [7]:
# Descripción del dataset para que el agente entienda el contexto
PREFIX = """Eres un agente analista de datos de ventas. Tienes acceso a un DataFrame de pandas
llamado `df` con datos de ventas de una empresa. Responde siempre en español.

Columnas del DataFrame:
- ordernumber    : ID único de cada orden
- quantityordered: unidades pedidas
- priceeach      : precio unitario en dólares
- sales          : ingreso total (cantidad × precio)
- orderdate      : fecha de la orden
- status         : estado (Shipped, Cancelled, etc.)
- productline    : línea de producto
- customername   : nombre del cliente
- country        : país del cliente
- dealsize       : tamaño del negocio (Small, Medium, Large)
"""

# Crear el agente
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type="openai-tools",
    verbose=True,                  # muestra el razonamiento del agente
    allow_dangerous_code=True,     # permite ejecutar código pandas
    prefix=PREFIX
)

print('✅ Agente pandas creado')

✅ Agente pandas creado


## CELDA 7 — Usar `invoke`

`invoke` es el método principal de LangChain para ejecutar el agente.

```python
# Estructura del invoke
resultado = agent.invoke({"input": "tu pregunta"})

# El resultado tiene dos campos:
resultado['input']   # tu pregunta original
resultado['output']  # la respuesta del agente
```

Con `verbose=True` también verás el razonamiento interno:
```
Thought:  → el agente piensa qué hacer
Action:   → decide usar la herramienta pandas
Action Input: → el código pandas que va a ejecutar
Observation: → el resultado del código
Final Answer: → la respuesta en lenguaje natural
```

In [8]:
# ── Ejemplo 1: pregunta simple ────────────────────────────────────
resultado = agent.invoke({"input": "¿Cuántas filas tiene el dataset y cuáles son sus columnas?"})
print('\n══ RESPUESTA ══')
print(resultado['output'])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape'}`
responded: Voy a verificar la cantidad de filas y las columnas del DataFrame `df`.

(2823, 25)
Invoking: `python_repl_ast` with `{'query': 'df.columns.tolist()'}`
responded: Voy a verificar la cantidad de filas y las columnas del DataFrame `df`.

['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'addressline2', 'city', 'state', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize']El dataset tiene **2,823 filas** (registros de ventas) y **25 columnas**. Estas son las columnas disponibles:

1. `ordernumber`: ID único de cada orden.
2. `quantityordered`: Unidades pedidas.
3. `priceeach`: Precio unitario en dólares.
4. `orderlinenumber`: Número de línea de la orden.
5. `sales`: Ingreso total (cantidad

In [9]:
# ── Ejemplo 2: análisis numérico ──────────────────────────────────
resultado = agent.invoke({"input": "¿Cuál es el total de ventas y el promedio por orden?"})
print('\n══ RESPUESTA ══')
print(resultado['output'])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Calcular el total de ventas (suma de la columna 'sales')\ntotal_ventas = df['sales'].sum()\n\n# Calcular el promedio de ventas por orden (promedio de la columna 'sales')\npromedio_por_orden = df['sales'].mean()\n\ntotal_ventas, promedio_por_orden"}`


(np.float64(10032628.85), np.float64(3553.889071909316))El **total de ventas** de la empresa es **$10,032,628.85**.

El **promedio de ventas por orden** es **$3,553.89**.

> Finished chain.

══ RESPUESTA ══
El **total de ventas** de la empresa es **$10,032,628.85**.

El **promedio de ventas por orden** es **$3,553.89**.


In [ ]:
# ── Ejemplo 3: comparativa ────────────────────────────────────────
resultado = agent.invoke({"input": "Compara las ventas por país y dime cuál lidera"})
print('\n══ RESPUESTA ══')
print(resultado['output'])

## CELDA 8 — Tu propio prompt libre
> Cambia la pregunta por lo que necesites analizar.

In [11]:
# ← Escribe aquí tu propia pregunta
mi_pregunta = "¿Cuál es el cliente con más compras y cuánto gastó en total?"

resultado = agent.invoke({"input": mi_pregunta})
print('\n══ RESPUESTA ══')
print(resultado['output'])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('customername')['sales'].sum().sort_values(ascending=False).head(1)"}`
responded: Voy a calcular el cliente con el mayor volumen de compras sumando el total de ventas (`sales`) por cada cliente.

customername
EURO SHOPPING CHANNEL    912294.11
Name: sales, dtype: float64El cliente con más compras es **EURO SHOPPING CHANNEL**, con un gasto total de **$912,294.11**.

> Finished chain.

══ RESPUESTA ══
El cliente con más compras es **EURO SHOPPING CHANNEL**, con un gasto total de **$912,294.11**.


## CELDA 9 — Chat interactivo con invoke
> Escribe `salir` para terminar.

In [ ]:
print('═══════════════════════════════════════')
print('  AGENTE PANDAS — LangChain + Mistral')
print('═══════════════════════════════════════')
print('Escribe cualquier pregunta sobre el dataset.')
print('Escribe "salir" para terminar.\n')

while True:
    entrada = input('Tú: ').strip()
    if not entrada:
        continue
    if entrada.lower() in ['salir', 'exit']:
        print('Agente: ¡Hasta luego!')
        break
    try:
        resultado = agent.invoke({"input": entrada})
        print(f'\nAgente: {resultado["output"]}\n')
        print('─' * 40)
    except Exception as e:
        print(f'\n⚠️ Error: {e}\n')

Configurar git

In [1]:
!git config --global user.email "pauortec@gmail.com"
!git config --global user.name "pauortec"
!git clone https://github.com/pauortec/IA_CLASES.git
print('✅ Repo clonado')

Cloning into 'IA_CLASES'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 2), reused 5 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 244.56 KiB | 16.30 MiB/s, done.
Resolving deltas: 100% (2/2), done.
✅ Repo clonado


Guardar el notebook RAG actual

In [2]:
from google.colab import _message
import json

nb = _message.blocking_request('get_ipynb', request='', timeout_sec=600)
with open('/content/IA_CLASES/rag_langchain_mistral.ipynb', 'w') as f:
    json.dump(nb['ipynb'], f)
print('✅ rag_langchain_mistral.ipynb guardado')

✅ rag_langchain_mistral.ipynb guardado


Subir notebook del agente pandas

In [3]:
import shutil, os

if os.path.exists('/content/langchain_pandas_agent.ipynb'):
    shutil.copy(
        '/content/langchain_pandas_agent.ipynb',
        '/content/IA_CLASES/langchain_pandas_agent.ipynb'
    )
    print('✅ langchain_pandas_agent.ipynb copiado')
else:
    print('⚠️  No se encontró langchain_pandas_agent.ipynb en /content')
    print('   Súbelo manualmente con files.upload() primero')

⚠️  No se encontró langchain_pandas_agent.ipynb en /content
   Súbelo manualmente con files.upload() primero
